# One-time setup: Snowflake catalog integrations

Run this manually, once, per environment — **not** part of the regular
`f1_ingest_and_transform_workflow` job. It creates one `CATALOG INTEGRATION` per
medallion zone, each pointing at that zone's own Unity Catalog catalog's Iceberg REST
endpoint (a REST integration's `CATALOG_NAME` is a single catalog, so three catalogs need
three integrations).

`sync_to_snowflake.ipynb` (the per-run job task) only manages the raw and curated zones'
`ICEBERG TABLE` objects against the first two integrations; it never (re)creates any
integration.

The analytics/gold zone is **not** synced table by table. A Snowflake catalog-linked
database over the `analytics` integration discovers and refreshes its tables on its own.
That database is deliberately **not** created here: this notebook runs as
`UC_ICEBERG_READER`, which holds no account-level `CREATE DATABASE`. A human creates it
once as `SYSADMIN` — see the "Create the catalog-linked database" step in the README. The
last cell below only lets `SYSADMIN` use the analytics integration for that.

## Before you run it: install the Snowflake connector

This notebook is **not** a job task, so it doesn't get the `snowflake_sync_env`
environment the sync job declares its dependency in
(`resources/job_definitions/f1_ingest_and_transform_workflow.yml`). Run manually on a
serverless cluster, `import snowflake.connector` (the first code cell) fails with
`ModuleNotFoundError` until you add the package yourself. Do this first, once per
session:

- **Environment side panel (recommended):** open the notebook's *Environment* panel, add
  `snowflake-connector-python` as a dependency, and apply it; or
- **A cell at the top:** `%pip install snowflake-connector-python` (serverless restarts
  Python for you).

Then run all cells.

In [ ]:
import snowflake.connector

dbutils.widgets.text("raw_catalog_name", "f1_raw")
dbutils.widgets.text("curated_catalog_name", "f1_curated")
dbutils.widgets.text("analytics_catalog_name", "f1_analytics")
dbutils.widgets.text("secret_scope", "snowflake-iceberg-sync")

raw_catalog_name = dbutils.widgets.get("raw_catalog_name")
curated_catalog_name = dbutils.widgets.get("curated_catalog_name")
analytics_catalog_name = dbutils.widgets.get("analytics_catalog_name")
secret_scope = dbutils.widgets.get("secret_scope")

catalog_names = [raw_catalog_name, curated_catalog_name, analytics_catalog_name]

# Auto-detect the current workspace host — no need to pass it manually.
databricks_host = spark.conf.get("spark.databricks.workspaceUrl")

print(f"Catalogs: {catalog_names}")
print(f"Databricks host: {databricks_host}")

In [ ]:
# All credentials come from the Databricks secret scope Terraform populates.
# Nothing sensitive is ever hard-coded here or passed as a plain job parameter.
snowflake_account = dbutils.secrets.get(secret_scope, "snowflake-account")
snowflake_user = dbutils.secrets.get(secret_scope, "snowflake-service-user")
snowflake_private_key_pem = dbutils.secrets.get(secret_scope, "snowflake-service-user-private-key")
snowflake_role = dbutils.secrets.get(secret_scope, "snowflake-role")
snowflake_warehouse = dbutils.secrets.get(secret_scope, "snowflake-warehouse")
snowflake_database = dbutils.secrets.get(secret_scope, "snowflake-database")
snowflake_target_schema = dbutils.secrets.get(secret_scope, "snowflake-schema")

databricks_sp_client_id = dbutils.secrets.get(secret_scope, "databricks-sp-client-id")
databricks_sp_client_secret = dbutils.secrets.get(secret_scope, "databricks-sp-client-secret")

In [ ]:
from cryptography.hazmat.primitives import serialization

# snowflake-connector-python wants the key as DER bytes, not the PEM string stored in the secret scope.
private_key_der = serialization.load_pem_private_key(
    snowflake_private_key_pem.encode(), password=None
).private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

conn = snowflake.connector.connect(
    account=snowflake_account,
    user=snowflake_user,
    private_key=private_key_der,
    role=snowflake_role,
    warehouse=snowflake_warehouse,
    database=snowflake_database,
    schema=snowflake_target_schema,
)
cur = conn.cursor()

In [ ]:
# Must match the formula sync_to_snowflake.ipynb uses to reference these integrations by
# name — both derive it from the catalog name alone, so nothing needs to be passed
# between the two notebooks.
def integration_name(catalog_name):
    return f"uc_{catalog_name}_iceberg"

oauth_token_uri = f"https://{databricks_host}/oidc/v1/token"
catalog_uri = f"https://{databricks_host}/api/2.1/unity-catalog/iceberg-rest"

# Syntax confirmed against Snowflake's "Configure a catalog integration for Unity
# Catalog" doc page — note CATALOG_SOURCE = ICEBERG_REST (one token, underscore, not
# two words), ACCESS_DELEGATION_MODE nested inside REST_CONFIG (not top-level), and
# OAUTH_TOKEN_URI/OAUTH_ALLOWED_SCOPES required for Databricks specifically (Databricks
# is not the OAuth token issuer by default, so it must be pointed at explicitly). No
# CATALOG_NAMESPACE here — these integrations have no default namespace, so
# sync_to_snowflake.ipynb sets CATALOG_NAMESPACE per table instead. VENDED_CREDENTIALS
# is also what lets the analytics zone's catalog-linked database work without an external volume.
# Still a Public Preview feature — re-verify against current docs if this starts failing.
for catalog_name in catalog_names:
    name = integration_name(catalog_name)
    cur.execute(f"""
        CREATE CATALOG INTEGRATION IF NOT EXISTS {name}
        CATALOG_SOURCE = ICEBERG_REST
        TABLE_FORMAT = ICEBERG
        REST_CONFIG = (
          CATALOG_URI = '{catalog_uri}'
          CATALOG_NAME = '{catalog_name}'
          ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
        )
        REST_AUTHENTICATION = (
          TYPE = OAUTH
          OAUTH_TOKEN_URI = '{oauth_token_uri}'
          OAUTH_CLIENT_ID = '{databricks_sp_client_id}'
          OAUTH_CLIENT_SECRET = '{databricks_sp_client_secret}'
          OAUTH_ALLOWED_SCOPES = ('all-apis')
        )
        ENABLED = TRUE
    """)
    print(f"Catalog integration '{name}' ready (catalog '{catalog_name}')")

In [ ]:
# Confirms each integration can actually reach Unity Catalog's REST endpoint and
# authenticate — separate from whether the CREATE statement itself parsed correctly.
for catalog_name in catalog_names:
    name = integration_name(catalog_name)
    cur.execute(f"SELECT SYSTEM$VERIFY_CATALOG_INTEGRATION('{name}')")
    print(f"{name}: {cur.fetchone()[0]}")

In [ ]:
# The analytics integration backs the catalog-linked database a human SYSADMIN creates
# (README: "Create the catalog-linked database"). SYSADMIN should already reach it through
# the role hierarchy (UC_ICEBERG_READER is granted to SYSADMIN, terraform/snowflake.tf), but
# CREATE DATABASE ... LINKED_CATALOG requires USAGE on the integration and that inheritance
# is not relied on here — grant it explicitly. This role owns the integration, so it may.
# Idempotent: safe to re-run.
analytics_integration = integration_name(analytics_catalog_name)
cur.execute(f"GRANT USAGE ON INTEGRATION {analytics_integration} TO ROLE SYSADMIN")
print(f"SYSADMIN may use '{analytics_integration}' — create the catalog-linked database next (README)")

In [ ]:
cur.close()
conn.close()